# Ch 13 — REINFORCE With / Without Baseline

**比较** REINFORCE + baseline 的方差降低效果——亲眼看 baseline 不引 bias 但显著降 variance。

## Setup
- 简化 CartPole（不用 gym，纯 numpy；用一个 2D 玩具任务模拟）
- Actor: 单层 softmax policy
- Baseline (with version): 简单 running average of returns
- 跑 200 episode × 50 seeds，比较 reward 曲线 + 方差

## 学什么
1. baseline 不改 expected gradient（数学保证）
2. baseline 显著降 update 方差
3. 收敛速度差异

## 链回
- [[ch13-policy-gradient]] §3.3 REINFORCE + Baseline
- [[_anki/ch13-pg-cards]] §C Card 10-13

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simple stochastic env:
# - state s ∈ [-1, 1] (uniform)
# - 2 actions: 0 = 'go left', 1 = 'go right'
# - reward if action matches sign of state (correct guess), else penalty
# - episode length = 10 steps, fresh s each step

def env_step(s):
    """Return reward depending on (s, a) via the policy chosen externally."""
    pass  # we'll handle inline

def softmax(x):
    e = np.exp(x - np.max(x))
    return e / e.sum()

def policy_prob(theta, s):
    # logits = [theta_0 + theta_1*s, theta_2 + theta_3*s]
    logits = np.array([theta[0] + theta[1]*s, theta[2] + theta[3]*s])
    return softmax(logits)

def grad_log_pi(theta, s, a):
    # ∇_θ log π(a|s)
    p = policy_prob(theta, s)
    g = np.zeros(4)
    g[2*a] = 1 - p[a]
    g[2*a + 1] = s * (1 - p[a])
    other = 1 - a
    g[2*other] = -p[other]
    g[2*other + 1] = -s * p[other]
    return g

print('Policy parameterized by θ ∈ R^4: logits = [θ_0 + θ_1·s, θ_2 + θ_3·s]')
print('Optimal: large positive θ_3, large negative θ_1 → match action to sign of state')

## 跑一个 episode 并算 return

In [ ]:
def run_episode(theta, horizon=10):
    states, actions, rewards = [], [], []
    for _ in range(horizon):
        s = np.random.uniform(-1, 1)
        probs = policy_prob(theta, s)
        a = np.random.choice(2, p=probs)
        # reward: +1 if a matches sign of s, -1 else
        correct = (a == 1 and s >= 0) or (a == 0 and s < 0)
        r = 1.0 if correct else -1.0
        states.append(s); actions.append(a); rewards.append(r)
    return states, actions, rewards

theta = np.array([0.0, 0.0, 0.0, 0.0])  # untrained
states, actions, rewards = run_episode(theta)
print(f'Untrained episode: total reward = {sum(rewards)} (random policy → ~0)')

## REINFORCE without baseline

Update: θ ← θ + α · G_t · ∇log π(a_t|s_t)

其中 G_t = Σ r_t（这里 γ=1）

In [ ]:
def reinforce_no_baseline(n_episodes=200, alpha=0.05, seed=0):
    np.random.seed(seed)
    theta = np.array([0.0, 0.0, 0.0, 0.0])
    returns = []
    for ep in range(n_episodes):
        states, actions, rewards = run_episode(theta)
        G = sum(rewards)
        returns.append(G)
        # use G_t (return-to-go) for each step
        for t in range(len(states)):
            G_t = sum(rewards[t:])
            theta = theta + alpha * G_t * grad_log_pi(theta, states[t], actions[t])
    return theta, returns

def reinforce_with_baseline(n_episodes=200, alpha=0.05, seed=0):
    np.random.seed(seed)
    theta = np.array([0.0, 0.0, 0.0, 0.0])
    returns = []
    baseline = 0.0
    for ep in range(n_episodes):
        states, actions, rewards = run_episode(theta)
        G = sum(rewards)
        returns.append(G)
        for t in range(len(states)):
            G_t = sum(rewards[t:])
            adv = G_t - baseline
            theta = theta + alpha * adv * grad_log_pi(theta, states[t], actions[t])
        # update baseline as running mean of episode returns
        baseline = 0.9 * baseline + 0.1 * G
    return theta, returns

theta_no, ret_no = reinforce_no_baseline()
theta_bl, ret_bl = reinforce_with_baseline()
print(f'No baseline  : final 50-ep avg return = {np.mean(ret_no[-50:]):.2f}, final θ = {np.round(theta_no, 2)}')
print(f'With baseline: final 50-ep avg return = {np.mean(ret_bl[-50:]):.2f}, final θ = {np.round(theta_bl, 2)}')

## 平均 50 个 seed 看方差差异

In [ ]:
n_seeds = 50
n_eps = 200
all_no = np.zeros((n_seeds, n_eps))
all_bl = np.zeros((n_seeds, n_eps))
for sd in range(n_seeds):
    _, r_no = reinforce_no_baseline(n_episodes=n_eps, seed=sd)
    _, r_bl = reinforce_with_baseline(n_episodes=n_eps, seed=sd)
    all_no[sd] = r_no
    all_bl[sd] = r_bl

mean_no, std_no = all_no.mean(axis=0), all_no.std(axis=0)
mean_bl, std_bl = all_bl.mean(axis=0), all_bl.std(axis=0)

x = np.arange(n_eps)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(x, mean_no, label='No baseline', color='C0')
axes[0].fill_between(x, mean_no - std_no, mean_no + std_no, color='C0', alpha=0.2)
axes[0].plot(x, mean_bl, label='With baseline', color='C1')
axes[0].fill_between(x, mean_bl - std_bl, mean_bl + std_bl, color='C1', alpha=0.2)
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Return (mean ± std across 50 seeds)')
axes[0].set_title('REINFORCE convergence: ± baseline')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(x, std_no, label='No baseline', color='C0')
axes[1].plot(x, std_bl, label='With baseline', color='C1')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Std of return across seeds')
axes[1].set_title('Update variance: baseline reduces std')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 关键观察

1. **baseline 不引 bias**——两者都收敛到相同 final return（最优 ~10）
2. **baseline 显著降 variance**——right panel 看 std 曲线
3. **baseline 收敛更快**——early episodes 上 with-baseline 上升更平稳

## 数学验证：baseline 为什么不引 bias

E_a[∇log π(a|s) · b(s)]
= b(s) · ∇_θ Σ_a π(a|s)
= b(s) · ∇_θ 1
= 0

→ 减 b(s) 等于减 0（in expectation）→ 不改 gradient 期望，但减 sample 之间的协方差。

## Interview 启示

- 手撕 baseline 不引 bias 的证明（3 行）
- 解释为什么 V_π(s) 是最优 baseline（minimize variance）
- baseline → critic → actor-critic → A2C → PPO 的演化链
- 跟 GAE 的关系：GAE 是 advantage 的 λ-加权版，比单 baseline 更精细

## 链回
- [[ch13-policy-gradient]] §3.3 / §3.4 Actor-Critic
- [[ch12-gae-step-by-step.ipynb]] — GAE 是 baseline + λ-mix